In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\admin\AppData\Local\Temp\ipykernel_11940\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [6]:
### Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" ok Loaded {len(documents)} pages")

        except Exception as e:
            print(f" x Error: {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 98 0 (offset 0)


Found 2 PDF files to process

Processing: 1.pdf
 ok Loaded 42 pages

Processing: 2.pdf
 ok Loaded 9 pages

 Total documents loaded: 51


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Mac OS X 10.7.5 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20140415211527Z00'00'", 'author': 'John Rueter', 'moddate': "D:20140415211527Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '..\\data\\pdf\\1.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': '1.pdf', 'file_type': 'pdf'}, page_content='Introduction to Solar Electricity'),
 Document(metadata={'producer': 'Mac OS X 10.7.5 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20140415211527Z00'00'", 'author': 'John Rueter', 'moddate': "D:20140415211527Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '..\\data\\pdf\\1.pdf', 'total_pages': 42, 'page': 1, 'page_label': '2', 'source_file': '1.pdf', 'file_type': 'pdf'}, page_content='PV Hands-on 1 \nTake a PV panel and a Digital multimeter \n(DMM) out in the sun \n \n1)\u202fFacing the sun, measure Voc and Isc  \n(careful about how to use DMM for \nVoltage vs Current!) \n2) At diff

In [9]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [10]:
chunks=split_documents(all_pdf_documents)
chunks

Split 51 documents into 63 chunks

 Example chunk:
Content: Introduction to Solar Electricity...
Metadata: {'producer': 'Mac OS X 10.7.5 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20140415211527Z00'00'", 'author': 'John Rueter', 'moddate': "D:20140415211527Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '..\\data\\pdf\\1.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': '1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Mac OS X 10.7.5 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20140415211527Z00'00'", 'author': 'John Rueter', 'moddate': "D:20140415211527Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '..\\data\\pdf\\1.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': '1.pdf', 'file_type': 'pdf'}, page_content='Introduction to Solar Electricity'),
 Document(metadata={'producer': 'Mac OS X 10.7.5 Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20140415211527Z00'00'", 'author': 'John Rueter', 'moddate': "D:20140415211527Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '..\\data\\pdf\\1.pdf', 'total_pages': 42, 'page': 1, 'page_label': '2', 'source_file': '1.pdf', 'file_type': 'pdf'}, page_content='PV Hands-on 1 \nTake a PV panel and a Digital multimeter \n(DMM) out in the sun \n \n1)\u202fFacing the sun, measure Voc and Isc  \n(careful about how to use DMM for \nVoltage vs Current!) \n2) At diff

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\workspace\python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10611.45it/s]


Model loaded successfully. Embedding dimension: 384


In [14]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 63 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.76it/s]

Generated embeddings with shape: (63, 384)
Adding 63 documents to vector store...
Successfully added 63 documents to vector store
Total documents in collection: 63


In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [18]:
rag_retriever

In [19]:
rag_retriever.retrieve("What are the classification ratings of solar panels?")

Retrieving documents for query: 'What are the classification ratings of solar panels?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.51it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5ee93384_56',
  'content': '50mph, while thin-film solar panels carry a lower rating due to their thin and flexible nature. \n \nHurricane rating \nWhile there is no formal solar classification rating for hurricanes, the Department of Energy recently \nexpanded its recommended design specifications for solar panels to safeguard against severe weather. \n \nThe new recommendations include:  \n \nModules with the highest ASTM E1830-15 rating for snow and wind loading in both the front and back. \nFasteners with true locking capability based on DIN 65151 standard \nThe use of through-bolting modules with locking fasteners instead of clamping fasteners',
  'metadata': {'content_length': 618,
   'producer': 'Microsoft® Word 2013',
   'page_label': '6',
   'creationdate': '2023-08-04T22:08:44+01:00',
   'source_file': '2.pdf',
   'total_pages': 9,
   'page': 5,
   'author': 'Microsoft account',
   'file_type': 'pdf',
   'doc_index': 56,
   'creator': 'Microsoft® Word 2013',
   '

In [20]:
rag_retriever.retrieve("What is the photovoltaic effect")

Retrieving documents for query: 'What is the photovoltaic effect'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 144.05it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_684a000e_3',
  'content': 'Photovoltaic Effect \n•\u202f PV cells produce electricity \nfrom sunlight (photons) \n•\u202f To work properly each cell \nneeds to receive sunlight \n•\u202fElectrical energy needs to \nbe stored for use if needed \nwhen there is not enough \nlight',
  'metadata': {'creator': 'PowerPoint',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\1.pdf',
   'page_label': '4',
   'doc_index': 3,
   'keywords': '',
   'producer': 'Mac OS X 10.7.5 Quartz PDFContext',
   'content_length': 227,
   'creationdate': "D:20140415211527Z00'00'",
   'total_pages': 42,
   'moddate': "D:20140415211527Z00'00'",
   'author': 'John Rueter',
   'source_file': '1.pdf',
   'aapl:keywords': '[]',
   'page': 3},
  'similarity_score': 0.47280246019363403,
  'distance': 0.527197539806366,
  'rank': 1},
 {'id': 'doc_392b4aa5_9',
  'content': 'Small Photovoltaic Systems',
  'metadata': {'creator': 'PowerPoint',
   'source': '..\\data\\pdf\\1.pdf',
   'source_file': '1.pdf',
 

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [26]:
from langchain_groq import ChatGroq
from langchain_core.prompts  import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [33]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = "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"
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [34]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key="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")
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


In [ ]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

In [38]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = "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"

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [39]:
answer=rag_simple("What are the types of solar radiation?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What are the types of solar radiation?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 170.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The types of solar radiation are: 

1. Direct light: Straight from the sun 
2. Diffuse light: Dispersed by clouds 
3. Reflected light: From snow, water, etc.


In [41]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What are the types of solar radiation?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What are the types of solar radiation?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 152.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The types of solar radiation are: 

1. Direct light: Straight from the sun 
2. Diffuse light: Dispersed by clouds 
3. Reflected light: From snow, water, etc.
Sources: [{'source': '1.pdf', 'page': 5, 'score': 0.3166695833206177, 'preview': 'The total radiation (sunlight) is comprised of:  \n          Direct light:  Straight from the sun \n           Diffuse light: Dispersed by clouds    \n                                                        Reflected light: From snow, water, etc. \n \n \n \n \n \nOn a completely cloudy day, all light may be ...'}, {'source': '2.pdf', 'page': 0, 'score': 0.11538630723953247, 'preview': 'The typical solar panel is composed of individual solar cells, each of which is made from layers of silicon, \nboron and phosphorus. The boron layer provides the positive charge, the phosphorus layer provides the \nnegative charge, and the silicon wafer acts as the semiconductor.  \n \nWhen the sun’s ph...'}, {'source': '2.pdf', 'page': 0, 'score': 0.1085487604

In [42]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What are the types of solar radiation?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What are the types of solar radiation?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 146.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
The total radiation (sunlight) is comprised of:  
          Direct light:  Straight from the sun 
           Diffuse light: Dispersed by clouds    
                                                        Reflected light: From snow, water, etc. 
 
 
 


 
 
On a completely cloudy day, all light may be diffuse. 
 
Most PV panels produce the most power in direct radiation. 
If one cell is shaded the panel electrical production (efficiency) drops drastically 
 
PV panels are much more sensitive to shade than thermal collectors 
Types of Solar Radiation

The typical solar panel is composed of individual solar cells, each of which is made from layers of silicon, 
boron and phosphorus. The boron layer provides the positive charge, the phosphorus layer provides the 
negative charge, and the silicon wafer acts as the semiconductor.  
 
When the sun’s photons strike the surface of the panel, it knocks out electrons from the silicon 
“sandwich” and into the electric field generated by the solar cells. This results in a directional current, 
which is then harnessed into usable power.  
 
solar module 
The entire process is called the photovoltaic effect, which is why solar panels are also known as 
photovoltaic panels or PV panels. A typical sol